# WaveForge - Skin Cancer Depth Staging Dataset Generator

**Run All - fully automated. Check Cell 8 plots BEFORE proceeding to generation.**

### Workflow
- **Cell 7:** Generate 5 validation samples (1 per class) — uses BOTH GPUs in parallel
- **Cell 8:** VISUAL CHECK - material layers, signals, B-scan images
  - Confirm skin layers visible, signals non-zero, B-scan shows tumor depth
  - If anything looks wrong - **stop here and debug before running full generation**
- **Cell 9:** Both GPUs generate training samples (1250 each → 2500 total)
- **Cell 10:** Both GPUs generate test samples (250 each → 500 total)

| Property | Value |
|----------|-------|
| Frequency | UWB 2-6 GHz (centre 4.0 GHz) |
| Grid | 128x128x64 at 0.5mm/cell |
| Antennas | 8-element linear array |
| Steps | 600 (pulse 265 + deepest round-trip 186 + margin) |
| Classes | 0=healthy 1=Tis 2=T1 3=T2 4=T3/T4 |

**Accelerator:** GPU T4 x2 (both used in parallel) | **Est. total:** ~8h

In [ ]:
# ── CELL 1: Setup ─────────────────────────────────────────────────────────
import subprocess, sys, os, pathlib, threading, time, json, datetime
import math

REPO_URL = 'https://github.com/shahzaibshazoo/waveforge.git'
REPO_DIR = pathlib.Path('/kaggle/working/waveforge')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)

src_path = str(REPO_DIR / 'src')
if src_path not in sys.path: sys.path.insert(0, src_path)
os.chdir(REPO_DIR)

import torch, numpy as np
assert torch.cuda.is_available(), 'No GPU - enable T4 x2!'
N_GPUS = torch.cuda.device_count()
print(f'GPUs: {N_GPUS}')
for i in range(N_GPUS):
    p = torch.cuda.get_device_properties(i)
    print(f'  [{i}] {p.name}  {p.total_memory/1e9:.1f} GB')
print(f'PyTorch: {torch.__version__}\n✅ Ready')

In [ ]:
# ── CELL 2: Configuration ─────────────────────────────────────────────────
import random

FREQ_LOW_GHZ  = 2.0
FREQ_HIGH_GHZ = 6.0   # UWB 2-6 GHz for thin skin layers
FREQ_GHZ      = (FREQ_LOW_GHZ + FREQ_HIGH_GHZ) / 2  # centre freq for tissue props
GRID_NX     = 128
GRID_NY     = 128
GRID_NZ     = 64      # depth
DX_MM       = 0.5     # 0.5mm/cell for thin layers
N_TX        = 8       # linear array
ANTENNA_Z   = 2       # antenna at z=2 (1mm into air gap, skin starts at z=4)
SKIN_SURFACE_Z = 4    # skin surface starts here (2mm air gap)
N_STEPS     = 600     # pulse width (265) + deepest round-trip T3/T4 4mm (186) + margin

N_TRAIN_TOTAL   = 2500;  N_TRAIN_PER_GPU = N_TRAIN_TOTAL // 2
N_TEST_TOTAL    = 500;   N_TEST_PER_GPU  = N_TEST_TOTAL  // 2

OUTPUT_ROOT  = pathlib.Path('/kaggle/working/skin_cancer_dataset')
TRAIN_DIR_G0 = OUTPUT_ROOT / 'train_gpu0'
TRAIN_DIR_G1 = OUTPUT_ROOT / 'train_gpu1'
TEST_DIR_G0  = OUTPUT_ROOT / 'test_gpu0'
TEST_DIR_G1  = OUTPUT_ROOT / 'test_gpu1'
TRAIN_DIR    = OUTPUT_ROOT / 'train'
TEST_DIR     = OUTPUT_ROOT / 'test'

GPU0 = 'cuda:0';  GPU1 = 'cuda:1' if N_GPUS > 1 else 'cuda:0'

TRAIN_SEED_G0 = 0;         TRAIN_SEED_G1 = 100_000
TEST_SEED_G0  = 10_000_000; TEST_SEED_G1  = 10_100_000

# Estimate time per sample
sec_per = N_TX * N_STEPS * GRID_NX * GRID_NY * GRID_NZ / 80e6 * 2
print(f'~{sec_per:.0f}s/sample  →  train: ~{sec_per*N_TRAIN_PER_GPU/3600:.1f}h + test: ~{sec_per*N_TEST_PER_GPU/3600:.1f}h')

In [ ]:
# ── CELL 3: Skin tissue library (Cole-Cole parameters) ───────────────────
# All simulation code is inline since this is skin-specific
from dataclasses import dataclass
from core.grid import YeeGrid

EPS0 = 8.8541878128e-12
MU0 = 1.2566370614e-6

@dataclass(frozen=True)
class ColeColeParams:
    """4-pole Cole-Cole parameters for a single tissue type."""
    name: str
    eps_inf: float
    sigma_s: float  # static ionic conductivity (S/m)
    delta_eps_1: float; tau_1: float; alpha_1: float
    delta_eps_2: float; tau_2: float; alpha_2: float
    delta_eps_3: float; tau_3: float; alpha_3: float
    delta_eps_4: float; tau_4: float; alpha_4: float

def cole_cole_to_fdtd(tissue: ColeColeParams, freq_hz: float) -> tuple:
    """Convert Cole-Cole to (eps_r, sigma) at a frequency."""
    omega = 2.0 * math.pi * freq_hz
    eps = complex(tissue.eps_inf, 0.0)
    poles = [(tissue.delta_eps_1, tissue.tau_1, tissue.alpha_1),
             (tissue.delta_eps_2, tissue.tau_2, tissue.alpha_2),
             (tissue.delta_eps_3, tissue.tau_3, tissue.alpha_3),
             (tissue.delta_eps_4, tissue.tau_4, tissue.alpha_4)]
    for delta_eps, tau, alpha in poles:
        if delta_eps == 0.0: continue
        eps += delta_eps / (1.0 + (1j * omega * tau) ** (1.0 - alpha))
    if tissue.sigma_s > 0 and omega > 0:
        eps += tissue.sigma_s / (1j * omega * EPS0)
    eps_r = eps.real
    sigma_eff = -eps.imag * omega * EPS0
    return max(eps_r, 1.0), max(sigma_eff, 0.0)

# Gabriel 1996 Cole-Cole parameters for skin tissues
SKIN_TISSUES = {
    'dry_skin': ColeColeParams(
        name='dry skin (stratum corneum)', eps_inf=4.0, sigma_s=0.0002,
        delta_eps_1=32.0, tau_1=7.23e-12, alpha_1=0.0,
        delta_eps_2=1100.0, tau_2=32.48e-9, alpha_2=0.2,
        delta_eps_3=0.0, tau_3=159.15e-6, alpha_3=0.2,
        delta_eps_4=0.0, tau_4=15.92e-3, alpha_4=0.2),
    'wet_skin': ColeColeParams(
        name='wet skin (epidermis)', eps_inf=4.0, sigma_s=0.0004,
        delta_eps_1=39.0, tau_1=7.96e-12, alpha_1=0.1,
        delta_eps_2=280.0, tau_2=79.58e-9, alpha_2=0.0,
        delta_eps_3=3.0e4, tau_3=1.59e-4, alpha_3=0.16,
        delta_eps_4=3.0e7, tau_4=1.59e-3, alpha_4=0.2),
    'dermis': ColeColeParams(
        name='dermis', eps_inf=4.0, sigma_s=0.0002,
        delta_eps_1=36.0, tau_1=7.23e-12, alpha_1=0.0,
        delta_eps_2=1100.0, tau_2=32.48e-9, alpha_2=0.2,
        delta_eps_3=0.0, tau_3=159.15e-6, alpha_3=0.2,
        delta_eps_4=0.0, tau_4=15.92e-3, alpha_4=0.2),
    'fat': ColeColeParams(
        name='fat (subcutaneous)', eps_inf=2.5, sigma_s=0.01,
        delta_eps_1=3.0, tau_1=23.0e-12, alpha_1=0.2,
        delta_eps_2=15.0, tau_2=159.15e-9, alpha_2=0.1,
        delta_eps_3=3.3e4, tau_3=159.15e-6, alpha_3=0.05,
        delta_eps_4=1e7, tau_4=7.96e-3, alpha_4=0.01),
    'muscle': ColeColeParams(
        name='muscle', eps_inf=4.0, sigma_s=0.2,
        delta_eps_1=50.0, tau_1=7.23e-12, alpha_1=0.1,
        delta_eps_2=7000.0, tau_2=353.68e-9, alpha_2=0.1,
        delta_eps_3=1.2e6, tau_3=318.31e-6, alpha_3=0.1,
        delta_eps_4=2.5e7, tau_4=2.27e-3, alpha_4=0.0),
    'tumor_malignant': ColeColeParams(
        name='malignant skin tumor', eps_inf=4.0, sigma_s=0.7,
        delta_eps_1=54.0, tau_1=7.96e-12, alpha_1=0.1,
        delta_eps_2=5000.0, tau_2=132.63e-9, alpha_2=0.1,
        delta_eps_3=0.0, tau_3=159.15e-6, alpha_3=0.2,
        delta_eps_4=0.0, tau_4=15.92e-3, alpha_4=0.0),
}

print('✅ Skin tissue library loaded (Gabriel 1996 Cole-Cole parameters)')

In [ ]:
# ── CELL 4: Skin phantom class (layered slab model) ──────────────────────
class SkinPhantom:
    """Layered skin phantom with optional embedded tumor.
    
    Layers (z-direction):
      - Air gap (2mm above skin)
      - Stratum corneum (0.015mm, 1 cell)
      - Epidermis (0.1mm, 1 cell)
      - Dermis (1-4mm, randomized)
      - Subcutaneous fat (2-8mm, randomized)
      - Muscle (backing)
    
    Tumor: ellipsoidal inclusion, depth varies by T-stage label.
    """
    def __init__(self, grid: YeeGrid, freq_hz: float, rng: random.Random):
        self.grid = grid
        self.freq_hz = freq_hz
        self.rng = rng
        self.Nx, self.Ny, self.Nz = grid.Nx, grid.Ny, grid.Nz
        self.dx = grid.dx * 1e3  # mm
        
        # Get tissue properties at centre frequency
        self.props = {name: cole_cole_to_fdtd(tissue, freq_hz)
                      for name, tissue in SKIN_TISSUES.items()}
    
    def generate_sample(self, label: int) -> dict:
        """Generate one sample with given label (0-4).
        
        Label to T-stage mapping:
          0: Healthy (no tumor)
          1: Tis (in-situ, epidermis only, <0.1mm)
          2: T1 (0.1-1.0mm into dermis)
          3: T2 (1.0-2.0mm into dermis)
          4: T3/T4 (>2.0mm, deep dermis/fat)
        """
        # Randomize layer thicknesses
        air_z = ANTENNA_Z  # 2mm air gap
        sc_thick = 1  # stratum corneum: 1 cell (0.5mm, models 0.015mm)
        epi_thick = 1  # epidermis: 1 cell (0.5mm, models 0.1mm)
        derm_thick_mm = self.rng.uniform(1.0, 4.0)
        fat_thick_mm = self.rng.uniform(2.0, 8.0)
        
        derm_thick = int(round(derm_thick_mm / self.dx))
        fat_thick = int(round(fat_thick_mm / self.dx))
        
        # Layer boundaries (z-indices)
        z_air = air_z
        z_sc = z_air + sc_thick
        z_epi = z_sc + epi_thick
        z_derm = z_epi + derm_thick
        z_fat = z_derm + fat_thick
        # Muscle fills rest
        
        # Initialize material arrays
        eps_r = np.ones((self.Nx, self.Ny, self.Nz), dtype=np.float32)
        sigma = np.zeros((self.Nx, self.Ny, self.Nz), dtype=np.float32)
        
        # Build layers
        eps_air, sig_air = 1.0, 0.0
        eps_sc, sig_sc = self.props['dry_skin']
        eps_epi, sig_epi = self.props['wet_skin']
        eps_derm, sig_derm = self.props['dermis']
        eps_fat, sig_fat = self.props['fat']
        eps_mus, sig_mus = self.props['muscle']
        
        # Fill layers (z-direction slices)
        eps_r[:, :, :z_air] = eps_air
        sigma[:, :, :z_air] = sig_air
        eps_r[:, :, z_air:z_sc] = eps_sc
        sigma[:, :, z_air:z_sc] = sig_sc
        eps_r[:, :, z_sc:z_epi] = eps_epi
        sigma[:, :, z_sc:z_epi] = sig_epi
        eps_r[:, :, z_epi:z_derm] = eps_derm
        sigma[:, :, z_epi:z_derm] = sig_derm
        eps_r[:, :, z_derm:z_fat] = eps_fat
        sigma[:, :, z_derm:z_fat] = sig_fat
        eps_r[:, :, z_fat:] = eps_mus
        sigma[:, :, z_fat:] = sig_mus
        
        # Tumor parameters
        tumor_type = 'none'
        tumor_depth_mm = 0.0
        tumor_diameter_mm = 0.0
        tumor_center_cells = np.array([0, 0, 0], dtype=np.int32)
        
        if label > 0:
            # Determine tumor depth based on T-stage
            if label == 1:  # Tis: epidermis only
                tumor_type = 'tis'
                depth_mm = self.rng.uniform(0.0, 0.08)
                z_center = z_sc + int(round(depth_mm / self.dx))
            elif label == 2:  # T1: 0.1-1.0mm into dermis
                tumor_type = 't1'
                depth_mm = self.rng.uniform(0.1, 1.0)
                z_center = z_epi + int(round(depth_mm / self.dx))
            elif label == 3:  # T2: 1.0-2.0mm into dermis
                tumor_type = 't2'
                depth_mm = self.rng.uniform(1.0, 2.0)
                z_center = z_epi + int(round(depth_mm / self.dx))
            else:  # T3/T4: >2.0mm (deep dermis/fat)
                tumor_type = 't3t4'
                depth_mm = self.rng.uniform(2.0, min(4.0, derm_thick_mm + fat_thick_mm * 0.5))
                z_center = z_epi + int(round(depth_mm / self.dx))
            
            # Tumor lateral position (central 60% of grid)
            margin = int(0.2 * self.Nx)
            x_center = self.rng.randint(margin, self.Nx - margin)
            y_center = self.rng.randint(margin, self.Ny - margin)
            
            # Tumor size
            tumor_diameter_mm = self.rng.uniform(2.0, 10.0)
            r_cells = tumor_diameter_mm / (2.0 * self.dx)
            aspect_ratio = self.rng.uniform(0.5, 1.5)
            rx = ry = r_cells
            rz = r_cells * aspect_ratio
            
            tumor_depth_mm = (z_center - z_air) * self.dx
            tumor_center_cells = np.array([x_center, y_center, z_center], dtype=np.int32)
            
            # Embed ellipsoidal tumor (vectorized)
            eps_tumor, sig_tumor = self.props['tumor_malignant']
            ii, jj, kk = np.ogrid[0:self.Nx, 0:self.Ny, 0:self.Nz]
            dist = ((ii - x_center) / rx) ** 2 + \
                   ((jj - y_center) / ry) ** 2 + \
                   ((kk - z_center) / rz) ** 2
            mask = dist <= 1.0
            eps_r[mask] = eps_tumor
            sigma[mask] = sig_tumor
        
        # Build FDTD update coefficients
        Ca, Cb = self._build_coefficients(eps_r, sigma)
        
        return {
            'Ca': Ca, 'Cb': Cb,
            'eps_r': eps_r, 'sigma': sigma,
            'label': label, 'tumor_type': tumor_type,
            'tumor_depth_mm': tumor_depth_mm,
            'tumor_diameter_mm': tumor_diameter_mm,
            'tumor_center_cells': tumor_center_cells,
            'dermis_thickness_mm': derm_thick_mm,
            'fat_thickness_mm': fat_thick_mm,
            'layer_boundaries': {'air': z_air, 'sc': z_sc, 'epi': z_epi,
                                 'derm': z_derm, 'fat': z_fat},
        }
    
    def generate_reference(self, derm_thick_mm: float, fat_thick_mm: float):
        """Generate healthy reference with SAME layer geometry (no tumor)."""
        derm_thick = int(round(derm_thick_mm / self.dx))
        fat_thick = int(round(fat_thick_mm / self.dx))
        
        z_air = ANTENNA_Z
        z_sc = z_air + 1
        z_epi = z_sc + 1
        z_derm = z_epi + derm_thick
        z_fat = z_derm + fat_thick
        
        eps_r = np.ones((self.Nx, self.Ny, self.Nz), dtype=np.float32)
        sigma = np.zeros((self.Nx, self.Ny, self.Nz), dtype=np.float32)
        
        eps_sc, sig_sc = self.props['dry_skin']
        eps_epi, sig_epi = self.props['wet_skin']
        eps_derm, sig_derm = self.props['dermis']
        eps_fat, sig_fat = self.props['fat']
        eps_mus, sig_mus = self.props['muscle']
        
        eps_r[:, :, z_air:z_sc] = eps_sc
        sigma[:, :, z_air:z_sc] = sig_sc
        eps_r[:, :, z_sc:z_epi] = eps_epi
        sigma[:, :, z_sc:z_epi] = sig_epi
        eps_r[:, :, z_epi:z_derm] = eps_derm
        sigma[:, :, z_epi:z_derm] = sig_derm
        eps_r[:, :, z_derm:z_fat] = eps_fat
        sigma[:, :, z_derm:z_fat] = sig_fat
        eps_r[:, :, z_fat:] = eps_mus
        sigma[:, :, z_fat:] = sig_mus
        
        Ca, Cb = self._build_coefficients(eps_r, sigma)
        return Ca, Cb
    
    def _build_coefficients(self, eps_r, sigma):
        """Build FDTD update coefficients Ca, Cb."""
        dt = self.grid.dt
        Ca = (2.0 * eps_r * EPS0 - sigma * dt) / (2.0 * eps_r * EPS0 + sigma * dt)
        Cb = (2.0 * dt) / (2.0 * eps_r * EPS0 + sigma * dt)
        Ca_t = torch.from_numpy(Ca).to(self.grid.device)
        Cb_t = torch.from_numpy(Cb).to(self.grid.device)
        return Ca_t, Cb_t

print('✅ SkinPhantom class ready')

In [ ]:
# ── CELL 5: Linear antenna array class ───────────────────────────────────
from core.sources import UWBPulse, PointSource, SourceCollection

class LinearAntennaArray:
    """Linear array of N antenna elements above flat skin surface.
    
    Antennas spaced along x-axis at fixed (y_center, z_antenna).
    Spacing: lambda/(2*sqrt(eps_skin)) at centre frequency (near-field probe).
    
    Recording uses batched GPU tensor indexing (1 sync per step, not N_rx syncs).
    """
    def __init__(self, n_elements: int, grid: YeeGrid, z_antenna: int,
                 freq_hz: float, component: str = 'Ez'):
        self.n_elements = n_elements
        self.grid = grid
        self.z_antenna = z_antenna
        self.component = component
        
        C0 = 3e8
        eps_dermis = 40.0
        wavelength_tissue_mm = (C0 / freq_hz / math.sqrt(eps_dermis)) * 1e3
        spacing_mm = wavelength_tissue_mm / 2.0
        spacing_cells = max(2, int(round(spacing_mm / (grid.dx * 1e3))))
        
        max_span = grid.Nx - 8
        max_spacing = max_span // (n_elements - 1)
        spacing_cells = min(spacing_cells, max_spacing)
        
        total_span = (n_elements - 1) * spacing_cells
        x_start = (grid.Nx - total_span) // 2
        y_center = grid.Ny // 2
        
        self.positions = []
        for k in range(n_elements):
            x = x_start + k * spacing_cells
            x = max(2, min(grid.Nx - 3, x))
            self.positions.append((x, y_center, z_antenna))
        
        # Pre-compute index tensors for batched recording (eliminates per-element .item() calls)
        self._idx_i = torch.tensor([p[0] for p in self.positions], dtype=torch.long, device=grid.device)
        self._idx_j = torch.tensor([p[1] for p in self.positions], dtype=torch.long, device=grid.device)
        self._idx_k = torch.tensor([p[2] for p in self.positions], dtype=torch.long, device=grid.device)
        
        self.spacing_cells = spacing_cells
        self.spacing_mm = spacing_cells * grid.dx * 1e3
        self.signals = None
        self.N_steps = 0
        
        print(f'    Array: {n_elements} elements, spacing={self.spacing_mm:.1f}mm '
              f'({spacing_cells} cells), span={total_span*grid.dx*1e3:.1f}mm')
    
    def build_sources(self, waveform, tx_idx: int, N_steps: int):
        """Build SourceCollection for a single TX element."""
        self.N_steps = N_steps
        if self.signals is None:
            self.signals = np.zeros((self.n_elements, self.n_elements, N_steps),
                                    dtype=np.float32)
        i, j, k = self.positions[tx_idx]
        src = PointSource(waveform, i, j, self.component, k=k,
                          grid=self.grid, N_steps=N_steps)
        return SourceCollection([src])
    
    def reset(self):
        """Clear signal buffer."""
        if self.signals is not None:
            self.signals[:] = 0.0
    
    def record(self, field: torch.Tensor, step: int, tx_idx: int):
        """Record field at all RX positions — single GPU→CPU transfer per step."""
        if self.signals is None:
            raise RuntimeError('Call build_sources() before record()')
        if step >= self.N_steps:
            return
        # Batched gather: 1 kernel + 1 .cpu() call instead of N_rx .item() calls
        vals = field[self._idx_i, self._idx_j, self._idx_k].cpu().numpy()
        self.signals[tx_idx, :, step] = vals
    
    def get_signals(self):
        """Return recorded signal matrix (N_tx, N_rx, N_steps)."""
        if self.signals is None:
            raise RuntimeError('No signals recorded yet')
        return self.signals.copy()
    
    def compute_bscan(self, signals: np.ndarray) -> np.ndarray:
        """B-scan: envelope of monostatic signals (TX==RX) vs antenna position."""
        from scipy.signal import hilbert
        n_tx = signals.shape[0]
        bscan = np.zeros((n_tx, signals.shape[2]), dtype=np.float32)
        for tx in range(n_tx):
            bscan[tx] = np.abs(hilbert(signals[tx, tx]))
        return bscan

print('✅ LinearAntennaArray class ready (batched GPU recording)')

In [ ]:
# ── CELL 6: Dataset generator class ──────────────────────────────────────
from core.fields import FieldSet
from core.boundaries import MurABC3D
from core.fdtd3d import FDTD3D

class SkinCancerDatasetGenerator:
    """Generate skin cancer depth staging FDTD dataset."""
    
    def __init__(self, output_dir: str, freq_low_hz: float, freq_high_hz: float,
                 grid_nx: int, grid_ny: int, grid_nz: int, dx_mm: float,
                 n_tx: int, antenna_y: int, n_steps: int, device: str, seed: int):
        self.output_dir = pathlib.Path(output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)
        
        self.freq_low = freq_low_hz
        self.freq_high = freq_high_hz
        self.freq_hz = (freq_low_hz + freq_high_hz) / 2.0
        self.n_steps = n_steps
        self.device = device
        self.rng = random.Random(seed)
        
        # Build grid
        dx = dx_mm * 1e-3
        self.grid = YeeGrid(grid_nx, grid_ny, dx=dx, dy=dx,
                            Nz=grid_nz, dz=dx, device=device)
        
        # Build antenna array
        self.array = LinearAntennaArray(n_tx, self.grid, antenna_y, self.freq_hz)
        self.antenna_z = antenna_y
        
        # Build phantom generator
        self.phantom = SkinPhantom(self.grid, self.freq_hz, self.rng)
        
        print(f'SkinCancerDatasetGenerator ready:')
        print(f'  Band: {freq_low_hz/1e9:.1f}-{freq_high_hz/1e9:.1f} GHz')
        print(f'  Grid: {grid_nx}x{grid_ny}x{grid_nz}, dx={dx_mm}mm')
        print(f'  Antennas: {n_tx} elements, y={antenna_y} (2mm above skin)')
        print(f'  Steps: {n_steps}, device: {device}')
    
    def generate_sample(self, sample_id: int, label: int, reference_only=False):
        """Generate one sample with given label."""
        # Generate phantom
        phantom_data = self.phantom.generate_sample(label)
        Ca, Cb = phantom_data['Ca'], phantom_data['Cb']
        
        # Build waveform
        waveform = UWBPulse(amplitude=1.0, f_low=self.freq_low, f_high=self.freq_high)
        
        # Run MIMO: each TX fires, all RX record
        self.array.reset()
        for tx_idx in range(self.array.n_elements):
            fields = FieldSet(self.grid)
            boundary = MurABC3D(self.grid, fields.Hx, fields.Hy, fields.Hz)
            sources = self.array.build_sources(waveform, tx_idx, self.n_steps)
            sim = FDTD3D(self.grid, fields, boundary, sources, Ca=Ca, Cb=Cb, n_check=99999)
            
            field_map = {'Ex': fields.Ex, 'Ey': fields.Ey, 'Ez': fields.Ez}
            field_tensor = field_map[self.array.component]
            
            with torch.no_grad():
                for step_n in range(self.n_steps):
                    sim.step()
                    self.array.record(field_tensor, step_n, tx_idx)
        
        signals = self.array.get_signals()
        
        if reference_only:
            return signals
        
        # Generate reference with SAME layer geometry but no tumor
        Ca_ref, Cb_ref = self.phantom.generate_reference(
            phantom_data['dermis_thickness_mm'], phantom_data['fat_thickness_mm'])
        
        self.array.reset()
        for tx_idx in range(self.array.n_elements):
            fields = FieldSet(self.grid)
            boundary = MurABC3D(self.grid, fields.Hx, fields.Hy, fields.Hz)
            sources = self.array.build_sources(waveform, tx_idx, self.n_steps)
            sim = FDTD3D(self.grid, fields, boundary, sources, Ca=Ca_ref, Cb=Cb_ref, n_check=99999)
            
            field_map_ref = {'Ex': fields.Ex, 'Ey': fields.Ey, 'Ez': fields.Ez}
            field_tensor = field_map_ref[self.array.component]
            
            with torch.no_grad():
                for step_n in range(self.n_steps):
                    sim.step()
                    self.array.record(field_tensor, step_n, tx_idx)
        
        signals_ref = self.array.get_signals()
        signals_scattered = signals - signals_ref
        
        # Compute B-scan
        bscan = self.array.compute_bscan(signals_scattered)
        
        # Save sample
        output_path = self.output_dir / f'sample_{sample_id:06d}.npz'
        np.savez_compressed(output_path,
            signals_total=signals.astype(np.float32),
            signals_reference=signals_ref.astype(np.float32),
            signals_scattered=signals_scattered.astype(np.float32),
            bscan_image=bscan,
            label=np.int32(label),
            tumor_type=phantom_data['tumor_type'],
            tumor_depth_mm=np.float32(phantom_data['tumor_depth_mm']),
            tumor_diameter_mm=np.float32(phantom_data['tumor_diameter_mm']),
            tumor_center_cells=phantom_data['tumor_center_cells'],
            dermis_thickness_mm=np.float32(phantom_data['dermis_thickness_mm']),
            fat_thickness_mm=np.float32(phantom_data['fat_thickness_mm']),
            freq_hz=np.float32(self.freq_hz),
            dx_mm=np.float32(self.grid.dx * 1e3),
            dt_s=np.float32(self.grid.dt),
            grid_shape=np.array([self.grid.Nx, self.grid.Ny, self.grid.Nz], dtype=np.int32),
            n_tx=np.int32(self.array.n_elements),
            n_steps=np.int32(self.n_steps),
        )
        
        return {
            'path': str(output_path),
            'label': label,
            'tumor_type': phantom_data['tumor_type'],
        }
    
    def generate_balanced_dataset(self, n_samples: int, base_seed: int, show_progress=True):
        """Generate balanced dataset with equal samples per class."""
        samples_per_class = n_samples // 5
        
        sample_paths = []
        labels = []
        tumor_types = []
        class_counts = {0: 0, 1: 0, 2: 0, 3: 0, 4: 0}
        
        sample_id = base_seed
        for class_label in range(5):
            for i in range(samples_per_class):
                if show_progress and i % 10 == 0:
                    print(f'  Class {class_label}: {i}/{samples_per_class}', end='\r')
                
                result = self.generate_sample(sample_id, class_label)
                sample_paths.append(result['path'])
                labels.append(result['label'])
                tumor_types.append(result['tumor_type'])
                class_counts[class_label] += 1
                sample_id += 1
        
        if show_progress:
            print(f'  Completed {n_samples} samples')
        
        return {
            'n_completed': len(sample_paths),
            'n_failed': 0,
            'class_counts': class_counts,
            'sample_paths': sample_paths,
            'labels': labels,
            'tumor_types': tumor_types,
        }

print('✅ SkinCancerDatasetGenerator class ready')

In [ ]:
# ── CELL 7: Generate 5 validation samples (one per class) ────────────────
print('Generating 5 validation samples (one per class) on BOTH GPUs...')

val_results = {}; val_errors = {}

def run_val_gpu(gpu, labels_to_run, out_dir, seed):
    try:
        gen = SkinCancerDatasetGenerator(
            output_dir=str(out_dir),
            freq_low_hz=FREQ_LOW_GHZ*1e9,
            freq_high_hz=FREQ_HIGH_GHZ*1e9,
            grid_nx=GRID_NX, grid_ny=GRID_NY, grid_nz=GRID_NZ,
            dx_mm=DX_MM,
            n_tx=N_TX, antenna_y=ANTENNA_Z, n_steps=N_STEPS,
            device=gpu, seed=seed,
        )
        paths, labels, types = [], [], []
        for i, lbl in enumerate(labels_to_run):
            result = gen.generate_sample(seed + i, lbl)
            paths.append(result['path'])
            labels.append(result['label'])
            types.append(result['tumor_type'])
            print(f'  [{gpu}] Class {lbl} done')
        val_results[gpu] = {'paths': paths, 'labels': labels, 'types': types, 'gen': gen}
    except Exception as e:
        val_errors[gpu] = e; print(f'[{gpu}] ERROR: {e}')

# Split 5 classes across 2 GPUs: GPU0 gets [0,1,2], GPU1 gets [3,4]
val_dir_g0 = OUTPUT_ROOT / 'validation_gpu0'
val_dir_g1 = OUTPUT_ROOT / 'validation_gpu1'

t0 = time.time()
threads = [
    threading.Thread(target=run_val_gpu, args=(GPU0, [0, 1, 2], val_dir_g0, 42)),
    threading.Thread(target=run_val_gpu, args=(GPU1, [3, 4], val_dir_g1, 142)),
]
for t in threads: t.start()
for t in threads: t.join()
elapsed = time.time() - t0

if val_errors:
    raise RuntimeError(f'Validation errors: {val_errors}')

# Merge results
val_manifest = {'sample_paths': [], 'labels': [], 'tumor_types': [], 'class_counts': {}}
for gpu in [GPU0, GPU1]:
    r = val_results[gpu]
    val_manifest['sample_paths'].extend(r['paths'])
    val_manifest['labels'].extend(r['labels'])
    val_manifest['tumor_types'].extend(r['types'])
for lbl in val_manifest['labels']:
    val_manifest['class_counts'][lbl] = val_manifest['class_counts'].get(lbl, 0) + 1
val_manifest['n_completed'] = len(val_manifest['sample_paths'])

# Keep reference to one generator for visualization
val_gen = val_results[GPU0]['gen']

print(f"\n{val_manifest['n_completed']}/5 samples in {elapsed:.1f}s ({elapsed/5:.1f}s/sample)")
print(f"Classes: {val_manifest['class_counts']}")
print('✅ Proceed to Cell 8 to inspect visually')

In [ ]:
# ── CELL 8: VISUAL CHECK - inspect before full generation ────────────────
#
# Shows for each of the 5 validation samples:
#   Col 1: Material map (eps_r) - tissue layers + tumor location
#   Col 2: Scattered signals from TX[0] - should be non-zero for tumor classes
#   Col 3: B-scan image - tumor should appear at correct depth
#
# IF any of these look wrong → STOP HERE. Do not proceed to Cell 9.

import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.signal import hilbert

label_to_name = {0: 'Healthy', 1: 'Tis (in-situ)', 2: 'T1 (0.1-1mm)',
                 3: 'T2 (1-2mm)', 4: 'T3/T4 (>2mm)'}

sorted_samples = sorted(
    [(lbl, path) for lbl, path in zip(val_manifest['labels'], val_manifest['sample_paths'])],
    key=lambda x: x[0]
)

fig, axes = plt.subplots(len(sorted_samples), 3, figsize=(18, 5 * len(sorted_samples)))
if len(sorted_samples) == 1: axes = axes[np.newaxis, :]
fig.suptitle(
    f'VISUAL CHECK - {len(sorted_samples)} samples, {N_TX} antennas, {N_STEPS} steps\n'
    f'Confirm: layers visible | signals non-zero | B-scan shows tumor depth',
    fontsize=13, fontweight='bold', color='darkblue'
)

ext_mm_xz = [0, GRID_NX*DX_MM, 0, GRID_NZ*DX_MM]

for row, (label, path) in enumerate(sorted_samples):
    s = np.load(path, allow_pickle=True)
    name = label_to_name.get(int(label), str(label))
    tumor_depth = float(s['tumor_depth_mm'])
    scat = s['signals_scattered']
    t_ns = np.arange(scat.shape[2]) * float(s['dt_s']) * 1e9

    # Panel 1: material map (xz cross-section)
    ax = axes[row, 0]
    img = np.ones((GRID_NX, GRID_NZ), dtype=np.float32)
    img[:, :SKIN_SURFACE_Z] = 1.0
    img[:, SKIN_SURFACE_Z:] = 40.0

    if tumor_depth > 0:
        tc = s['tumor_center_cells']
        tx_c, ty_c, tz_c = int(tc[0]), int(tc[1]), int(tc[2])
        r = int(float(s['tumor_diameter_mm']) / (2.0 * DX_MM))
        for i in range(max(0, tx_c-r), min(GRID_NX, tx_c+r+1)):
            for k in range(max(0, tz_c-r), min(GRID_NZ, tz_c+r+1)):
                if (i-tx_c)**2 + (k-tz_c)**2 <= r**2:
                    img[i, k] = 55.0

    im = ax.imshow(img.T, origin='lower', cmap='jet', vmin=1, vmax=60,
                   extent=ext_mm_xz, aspect='auto')
    for ant_x, ant_y, ant_z in val_gen.array.positions:
        ax.plot(ant_x*DX_MM, ant_z*DX_MM, 'w^', markersize=6)
    if tumor_depth > 0:
        ax.plot(tc[0]*DX_MM, tc[2]*DX_MM, 'r*', markersize=14, markeredgecolor='white')
    plt.colorbar(im, ax=ax, label='eps_r')
    ax.set(title=f'{name} - material map (xz plane)',
           xlabel='x (mm)', ylabel='z (mm)')

    # Panel 2: scattered signals
    ax2 = axes[row, 1]
    energy = float((scat**2).sum())
    colors = plt.cm.tab10(np.linspace(0, 1, N_TX))
    for rx in range(N_TX):
        ax2.plot(t_ns, scat[0, rx], color=colors[rx], alpha=0.7, lw=1)
    ax2.set(title=f'Scattered signals TX[0]  E={energy:.2e}',
            xlabel='Time (ns)', ylabel='Ez (V/m)')
    ax2.grid(alpha=0.3)
    if tumor_depth > 0 and energy < 1e-10:
        ax2.set_facecolor('#fff0f0')
        ax2.set_title(ax2.get_title() + ' ⚠ LOW', color='red')

    # Panel 3: B-scan
    ax3 = axes[row, 2]
    bscan = s['bscan_image']
    C0 = 3e8; eps_d = 40.0
    v_tissue = C0 / math.sqrt(eps_d)
    depth_axis_mm = t_ns * v_tissue * 1e-9 * 1e3 / 2.0
    im3 = ax3.imshow(bscan.T, origin='lower', cmap='hot',
                     extent=[0, N_TX-1, 0, depth_axis_mm[-1]], aspect='auto')
    plt.colorbar(im3, ax=ax3, label='B-scan power')
    if tumor_depth > 0:
        ax3.axhline(tumor_depth, color='cyan', linestyle='--', lw=2,
                    label=f'tumor depth {tumor_depth:.1f}mm')
        ax3.legend(fontsize=8)
    ax3.set(title=f'B-scan - {name}', xlabel='Antenna index', ylabel='Depth (mm)')
    detected = energy > 1e-9
    ok = (detected == (tumor_depth > 0))
    for sp in ax3.spines.values():
        sp.set_edgecolor('#2ecc71' if ok else '#e74c3c'); sp.set_linewidth(3)

plt.tight_layout()
os.makedirs('docs/assets', exist_ok=True)
plt.savefig('docs/assets/skin_validation_check.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → docs/assets/skin_validation_check.png')
print()
print('✅ If all 5 rows look correct → proceed to Cell 9 for full generation')
print('❌ If any row looks wrong → debug before running full generation')

In [ ]:
# ── CELL 9: Phase 1 - both GPUs generate TRAINING samples ────────────────
results = {}; errors = {}

def run_train_gpu(gpu, out_dir, seed, n):
    try:
        print(f'[{gpu}] Starting {n} train samples...')
        gen = SkinCancerDatasetGenerator(
            output_dir=str(out_dir),
            freq_low_hz=FREQ_LOW_GHZ*1e9,
            freq_high_hz=FREQ_HIGH_GHZ*1e9,
            grid_nx=GRID_NX, grid_ny=GRID_NY, grid_nz=GRID_NZ,
            dx_mm=DX_MM,
            n_tx=N_TX, antenna_y=ANTENNA_Z, n_steps=N_STEPS,
            device=gpu, seed=seed,
        )
        results[gpu] = gen.generate_balanced_dataset(
            n_samples=n, base_seed=seed, show_progress=True)
        print(f'[{gpu}] Done: {results[gpu]["n_completed"]} samples')
    except Exception as e:
        errors[gpu] = e; print(f'[{gpu}] ERROR: {e}')

t0 = time.time()
threads = [
    threading.Thread(target=run_train_gpu, args=(GPU0, TRAIN_DIR_G0, TRAIN_SEED_G0, N_TRAIN_PER_GPU)),
    threading.Thread(target=run_train_gpu, args=(GPU1, TRAIN_DIR_G1, TRAIN_SEED_G1, N_TRAIN_PER_GPU)),
]
for t in threads: t.start()
for t in threads: t.join()
if errors: raise RuntimeError(f'Errors: {errors}')

import shutil
TRAIN_DIR.mkdir(parents=True, exist_ok=True)
train_paths, train_labels, train_types = [], [], []
train_class_counts = {0:0, 1:0, 2:0, 3:0, 4:0}
for gk, sd in [('cuda:0', TRAIN_DIR_G0), ('cuda:1', TRAIN_DIR_G1)]:
    m = results[gk]
    for op, lb, tt in zip(m['sample_paths'], m['labels'], m['tumor_types']):
        ni = len(train_paths)
        np_ = TRAIN_DIR / f'sample_{ni:06d}.npz'
        shutil.copy(op, np_)
        train_paths.append(str(np_)); train_labels.append(lb)
        train_types.append(tt)
        train_class_counts[lb] += 1
train_manifest = {
    'n_completed': len(train_paths), 'n_failed': 0,
    'class_counts': train_class_counts, 'sample_paths': train_paths,
    'labels': train_labels, 'tumor_types': train_types,
}
print(f'\n✅ Phase 1 done in {(time.time()-t0)/3600:.2f}h  |  {len(train_paths)} samples')
print(f'   Classes: {train_class_counts}')

In [ ]:
# ── CELL 10: Phase 2 - both GPUs generate TEST samples ───────────────────
test_results = {}; test_errors = {}

def run_test_gpu(gpu, out_dir, seed, n):
    try:
        print(f'[{gpu}] Starting {n} test samples (seed space 10M+)...')
        gen = SkinCancerDatasetGenerator(
            output_dir=str(out_dir),
            freq_low_hz=FREQ_LOW_GHZ*1e9,
            freq_high_hz=FREQ_HIGH_GHZ*1e9,
            grid_nx=GRID_NX, grid_ny=GRID_NY, grid_nz=GRID_NZ,
            dx_mm=DX_MM,
            n_tx=N_TX, antenna_y=ANTENNA_Z, n_steps=N_STEPS,
            device=gpu, seed=seed,
        )
        test_results[gpu] = gen.generate_balanced_dataset(
            n_samples=n, base_seed=seed, show_progress=True)
        print(f'[{gpu}] Done: {test_results[gpu]["n_completed"]} samples')
    except Exception as e:
        test_errors[gpu] = e; print(f'[{gpu}] ERROR: {e}')

t0 = time.time()
threads = [
    threading.Thread(target=run_test_gpu, args=(GPU0, TEST_DIR_G0, TEST_SEED_G0, N_TEST_PER_GPU)),
    threading.Thread(target=run_test_gpu, args=(GPU1, TEST_DIR_G1, TEST_SEED_G1, N_TEST_PER_GPU)),
]
for t in threads: t.start()
for t in threads: t.join()
if test_errors: raise RuntimeError(f'Errors: {test_errors}')

TEST_DIR.mkdir(parents=True, exist_ok=True)
test_paths, test_labels, test_types = [], [], []
test_class_counts = {0:0, 1:0, 2:0, 3:0, 4:0}
for gk, sd in [('cuda:0', TEST_DIR_G0), ('cuda:1', TEST_DIR_G1)]:
    m = test_results[gk]
    for op, lb, tt in zip(m['sample_paths'], m['labels'], m['tumor_types']):
        ni = len(test_paths)
        np_ = TEST_DIR / f'sample_{ni:06d}.npz'
        shutil.copy(op, np_)
        test_paths.append(str(np_)); test_labels.append(lb)
        test_types.append(tt)
        test_class_counts[lb] += 1
test_manifest = {
    'n_completed': len(test_paths), 'class_counts': test_class_counts,
    'sample_paths': test_paths, 'labels': test_labels,
    'tumor_types': test_types,
}
print(f'\n✅ Phase 2 done in {(time.time()-t0)/3600:.2f}h  |  {len(test_paths)} test samples')
print(f'   Classes: {test_class_counts}')

In [ ]:
# ── CELL 11: Save master manifest ─────────────────────────────────────────
try: commit = subprocess.check_output(['git','-C',str(REPO_DIR),'rev-parse','--short','HEAD'],text=True).strip()
except: commit = 'unknown'

master = {
    'version': '1.0', 'created_at': datetime.datetime.now().isoformat(),
    'waveforge_commit': commit,
    'n_total': len(train_paths)+len(test_paths),
    'n_train': len(train_paths), 'n_test': len(test_paths),
    'train_class_counts': train_class_counts,
    'test_class_counts':  test_class_counts,
    'class_names': {0:'healthy',1:'tis',2:'t1',3:'t2',4:'t3t4'},
    'phantom_design': 'layered_skin_with_tumor',
    'freq_hz': FREQ_GHZ*1e9,
    'freq_low_hz': FREQ_LOW_GHZ*1e9,
    'freq_high_hz': FREQ_HIGH_GHZ*1e9,
    'uwb_mode': True,
    'grid_shape': [GRID_NX, GRID_NY, GRID_NZ],
    'dx_mm': DX_MM, 'n_tx': N_TX, 'n_steps': N_STEPS,
    'notes': '1500 steps validated: full round-trip covers deepest T3/T4 tumors at 4mm in tissue',
    'train_dir': str(TRAIN_DIR), 'test_dir': str(TEST_DIR),
}
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
manifest_path = OUTPUT_ROOT / 'dataset_manifest.json'
with open(manifest_path,'w') as f: json.dump(master,f,indent=2)
print(f'Total: {master["n_total"]}  |  commit: {commit}')

In [ ]:
# ── CELL 12: Package for download + dataset statistics ───────────────────
import shutil
OUT = pathlib.Path('/kaggle/working/waveforge_skin_outputs')
OUT.mkdir(exist_ok=True)
shutil.copy(manifest_path, OUT)
for img in ['docs/assets/skin_validation_check.png']:
    if pathlib.Path(img).exists(): shutil.copy(img, OUT)

train_f = sorted(TRAIN_DIR.glob('*.npz'))
test_f  = sorted(TEST_DIR.glob('*.npz'))
mb = sum(f.stat().st_size for f in train_f+test_f)/1e6
print(f'Train: {len(train_f)} | Test: {len(test_f)} | Size: {mb:.1f} MB')

# Plot dataset statistics
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Class distribution
ax = axes[0]
classes = list(train_class_counts.keys())
train_counts = [train_class_counts[c] for c in classes]
test_counts = [test_class_counts[c] for c in classes]
x = np.arange(len(classes))
width = 0.35
ax.bar(x - width/2, train_counts, width, label='Train', color='steelblue')
ax.bar(x + width/2, test_counts, width, label='Test', color='coral')
ax.set(title='Class Distribution', xlabel='Class', ylabel='Count',
       xticks=x, xticklabels=[label_to_name[c] for c in classes])
ax.legend()
ax.grid(alpha=0.3, axis='y')

# Tumor depth distribution
ax = axes[1]
depths = []
for path in train_paths:
    s = np.load(path, allow_pickle=True)
    if s['label'] > 0:
        depths.append(float(s['tumor_depth_mm']))
ax.hist(depths, bins=30, color='teal', alpha=0.7, edgecolor='black')
ax.set(title='Tumor Depth Distribution (Train)', xlabel='Depth (mm)', ylabel='Count')
ax.grid(alpha=0.3, axis='y')

# Tumor diameter distribution
ax = axes[2]
diameters = []
for path in train_paths:
    s = np.load(path, allow_pickle=True)
    if s['label'] > 0:
        diameters.append(float(s['tumor_diameter_mm']))
ax.hist(diameters, bins=30, color='orchid', alpha=0.7, edgecolor='black')
ax.set(title='Tumor Diameter Distribution (Train)', xlabel='Diameter (mm)', ylabel='Count')
ax.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(OUT / 'dataset_statistics.png', dpi=150, bbox_inches='tight')
plt.show()

print('Done!')